In [ ]:
from pyspark.sql import SparkSession

# 1) إنهاء أي جلسة قديمة معلقة احتياطياً
if 'spark' in locals():
    spark.stop()

# 2) بناء جلسة نظيفة بالكامل مع تحميل حزمة Postgres
spark = SparkSession.builder \
    .appName("Check Postgres Tables Direct") \
    .master("local[*]") \
    .config("spark.jars.packages", "org.postgresql:postgresql:42.6.0") \
    .config("spark.eventLog.enabled", "false") \
    .config("spark.eventLog.dir", "/tmp") \
    .getOrCreate()

# 3) محاولة الاتصال بـ Postgres وقراءة الجداول
try:
    tables_df = spark.read \
        .format("jdbc") \
        .option("url", "jdbc:postgresql://postgres:5432/smart_agriculture") \
        .option("user", "postgres") \
        .option("password", "root") \
        .option("driver", "org.postgresql.Driver") \
        .option("dbtable", "(SELECT table_name FROM information_schema.tables WHERE table_schema = 'public') AS tables") \
        .load()

    print("🎉 ممتاز! تم الاتصال بنجاح. الجداول الموجودة لديك حالياً في قاعدة البيانات هي:")
    tables_df.show()
except Exception as e:
    print("❌ خطأ أثناء جلب البيانات: ", e)